# Policy evaluation — RL vs recipe/PID on held-out seeds

Evaluates one or many **training runs** of the single-phase MC-PILCO agent.

Design rules this notebook follows:

- **Paired.** Every RL batch is matched by a recipe batch on the *same* simulator seed, so the
  batch-to-batch realisation cancels and the difference is attributable to the controller. The
  recipe arm is `pid_baseline=True`, which sets `Fs` to the recipe profile while every other stream
  is already the recipe in both arms — so the two arms differ **only** in `Fs`.
- **Held out.** Evaluation seeds are far from any training realisation, and the seeds used for the
  headline result are **disjoint** from those used for the progression curves, so no seed is both
  inspected and reported on.
- **The training run is the unit of analysis.** A paired test over evaluation seeds measures
  *batch* variance and answers "does this policy beat the recipe on new batches". It cannot answer
  "does the method beat the recipe", because training-run variance never enters it. With several
  runs, E.2 aggregates each run to a single number and tests across runs — that is the statistic
  that supports a claim about the method.
- **Mean is not enough.** Yields here are bimodal: healthy batches sit near 3500 kg while collapsed
  ones fall near 1000 kg, and a mean describes neither. E.4 reports collapse rate, worst case and
  tail risk alongside the mean.

## 0. Setup

In [ ]:
%matplotlib inline
import os, sys, json, pickle, contextlib, io, glob, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy import stats

import os as _os, sys as _sys
_ROOT = _os.path.dirname(_os.path.abspath(""))
_sys.path.insert(0, _ROOT)
_sys.path.insert(0, _os.path.dirname(_ROOT))

from mcpilco.config_single_phase import get_config
from mcpilco.pensim_wrapper import (PenSimWrapper, PenSimMCPILCO, STATE_NAMES, STATE_DIM,
                                    T_SAMPLING, CONTROL_H, VISC_MAX)
from experiments.eval_utils import yield_kg, constraint_diagnostics

print("state:", STATE_NAMES)

## 1. Config

`RUN_GLOBS` selects which training runs to evaluate — point it at the campaign directory to compare
arms, or at a single run to inspect one.

`EVAL_SEEDS` and `PROG_SEEDS` are deliberately disjoint. Re-using the reporting seeds to draw the
learning curve would mean the same batches both guide inspection and produce the headline number.

Results are cached to CSV: evaluating every trial of every run is thousands of simulator batches,
and nothing here should be re-run just to redraw a figure. Delete the cache file to force a refresh.

In [ ]:
RUN_GLOBS = [
    "results/cluster/full/*", "results/single_phase/seed3_1", "results/single_phase/seed2_24",
    # "results/single_phase/seed3_*",  # or a local run
]

EVAL_SEEDS = [700000 + i for i in range(3)]   # headline paired comparison
PROG_SEEDS = [710000 + i for i in range(2)]    # per-trial progression (disjoint, kept small)

FAILED_KG = 1500.0        # a batch below this has collapsed, not merely underperformed
CACHE = Path(_ROOT) / "results" / "evaluate_policy_cache.csv"

assert not (set(EVAL_SEEDS) & set(PROG_SEEDS)), "eval and progression seeds must be disjoint"

RUN_DIRS = sorted({p for g in RUN_GLOBS for p in glob.glob(os.path.join(_ROOT, g))
                   if os.path.isfile(os.path.join(p, "log.pkl"))})
print(f"{len(RUN_DIRS)} run(s) found")
for p in RUN_DIRS:
    print("   ", os.path.relpath(p, _ROOT))
if not RUN_DIRS:
    print("\nNo runs matched RUN_GLOBS -- edit it above.")

## 2. Helpers

`_run_meta` reads `note.txt` so each run carries its own arm label; this is why the training script
records every parameter there rather than relying on directory names.

`_load_agent` builds **one** agent and swaps policy weights into it. `get_config` measures the
initial state by rolling recipe batches, so constructing an agent per run would be needlessly slow.

The `centers` guard matters: a policy trained on a different `STATE_DIM` cannot be loaded into the
current one. Without the check `load_state_dict` raises a shape error mid-loop; with it, stale runs
are skipped with a message naming the fix.

In [ ]:
def _run_meta(run_dir):
    '''Arm label + recorded parameters, parsed from note.txt.'''
    meta = {"run": os.path.basename(run_dir), "path": run_dir}
    note = Path(run_dir) / "note.txt"
    if note.exists():
        for line in note.read_text().splitlines():
            m = re.match(r"^(seed|num_trials|risk_weight|visc_penalty|harvest_reward) = (.+)$", line)
            if m:
                meta[m.group(1)] = m.group(2).strip()
    lg = pickle.load(open(Path(run_dir) / "log.pkl", "rb"))
    meta["n_trials"] = len(lg.get("parameters_trial_list", []))
    meta["policy_state_dim"] = int(lg["parameters_trial_list"][0]["centers"].shape[1])
    # Short arm label from the cost-shaping flags, so figures are readable without the note.
    vp = float(meta.get("visc_penalty", "nan") or "nan")
    hv = str(meta.get("harvest_reward", "?"))
    rw = float(meta.get("risk_weight", "nan") or "nan")
    meta["arm"] = f"vp={vp:g} hv={hv[0]} rw={rw:g}" if np.isfinite(vp) else meta["run"]
    return meta


_agent = None
def _load_policy(run_dir, trial):
    '''Numpy policy for `trial` (1-based) of `run_dir`, reusing a single agent instance.'''
    global _agent
    if _agent is None:
        with contextlib.redirect_stdout(io.StringIO()):
            cfg = get_config(seed=0, num_trials=1)
            _agent = PenSimMCPILCO(pensim_wrapper=PenSimWrapper(), **cfg["mc_pilco_init"])
    with contextlib.redirect_stdout(io.StringIO()):
        _agent.load_policy_from_log(num_trial=trial, folder=str(run_dir).rstrip("/") + "/")
    return _agent.control_policy.get_np_policy()


_wrapper = PenSimWrapper()
def _batch(seed, policy=None, pid_baseline=False):
    '''One simulator batch; returns the metrics we score on.'''
    with contextlib.redirect_stdout(io.StringIO()):
        _wrapper.rollout(None, policy, CONTROL_H, T_SAMPLING, 0,
                         seed=seed, pid_baseline=pid_baseline)
    m = _wrapper.monitor[-1]
    d = constraint_diagnostics(m)
    return {"yield_kg": yield_kg(m),
            "max_visc": float(np.max(m["Viscosity"])),
            "final_P": d["final_P"],
            "max_Wt": d["max_Wt"],
            "sugar_kg": float(np.sum(m["Fs"])) * (m["t"][1] - m["t"][0]) * 1.324 / 1000.0}


_cache = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame()
def _cached(rows_wanted, compute):
    '''Evaluate only the (key) rows not already in the on-disk cache.'''
    global _cache
    key = ["run", "trial", "seed", "kind"]
    have = set(map(tuple, _cache[key].values)) if len(_cache) else set()
    todo = [r for r in rows_wanted if tuple(r[k] for k in key) not in have]
    if todo:
        print(f"computing {len(todo)} new batches ({len(rows_wanted) - len(todo)} cached)...")
        new = pd.DataFrame([{**r, **compute(r)} for r in todo])
        _cache = pd.concat([_cache, new], ignore_index=True) if len(_cache) else new
        _cache.to_csv(CACHE, index=False)
    else:
        print(f"all {len(rows_wanted)} batches cached")
    sel = _cache.set_index(key).loc[[tuple(r[k] for k in key) for r in rows_wanted]]
    return sel.reset_index()


RUNS = [_run_meta(p) for p in RUN_DIRS]
_stale = [r for r in RUNS if r["policy_state_dim"] != STATE_DIM]
for r in _stale:
    print(f"SKIP {r['run']}: policy has state_dim={r['policy_state_dim']}, current STATE_DIM="
          f"{STATE_DIM}. Re-train this run on the current state to evaluate it.")
RUNS = [r for r in RUNS if r["policy_state_dim"] == STATE_DIM]
print(f"\n{len(RUNS)} evaluable run(s)")
pd.DataFrame(RUNS)[["run", "arm", "seed", "n_trials"]] if RUNS else None

## E.1 — Held-out paired comparison: recipe/PID vs RL

The final policy of each run, rolled across `EVAL_SEEDS`, each seed matched by a recipe batch.

The recipe arm is computed **once** and shared by every run, which is what makes the arms comparable
to each other as well as to the baseline.

In [ ]:
# Recipe baseline: once per seed, shared by every run.
recipe_rows = [{"run": "_recipe", "trial": 0, "seed": s, "kind": "recipe"} for s in EVAL_SEEDS]
recipe_df = _cached(recipe_rows, lambda r: _batch(r["seed"], policy=None, pid_baseline=True))
recipe_y = recipe_df.set_index("seed")["yield_kg"]

rows = [{"run": r["run"], "trial": r["n_trials"], "seed": s, "kind": "rl"}
        for r in RUNS for s in EVAL_SEEDS]
_paths = {r["run"]: r["path"] for r in RUNS}
rl_df = _cached(rows, lambda r: _batch(r["seed"], policy=_load_policy(_paths[r["run"]], r["trial"])))

rl_df["yield_recipe"] = rl_df["seed"].map(recipe_y)
rl_df["delta"] = rl_df["yield_kg"] - rl_df["yield_recipe"]
rl_df["arm"] = rl_df["run"].map({r["run"]: r["arm"] for r in RUNS})
rl_df["train_seed"] = rl_df["run"].map({r["run"]: r.get("seed", "?") for r in RUNS})

print(f"recipe on held-out seeds: mean {recipe_y.mean():.0f} kg, sd {recipe_y.std(ddof=1):.0f}, "
      f"worst {recipe_y.min():.0f}\n")
rl_df[["run", "arm", "seed", "yield_kg", "yield_recipe", "delta", "max_visc"]].round(1)

## E.2 — Statistics

Two levels, and the distinction is the point:

- **Per run** — paired Wilcoxon over `EVAL_SEEDS`. Answers "does *this* policy beat the recipe on
  new batches". Wilcoxon rather than a t-test is primary because collapsed batches make the delta
  distribution bimodal, which violates normality; the t-test is shown for reference only.
- **Across runs** — each run collapses to one number (its mean paired delta), then a test across
  runs. This is the only statistic that supports a claim about the *method*, and it needs several
  training seeds per arm to say anything at all.

In [ ]:
per_run = []
for r in RUNS:
    d = rl_df[rl_df.run == r["run"]]
    dl = d["delta"].values
    try:
        w_p = float(stats.wilcoxon(d["yield_kg"], d["yield_recipe"]).pvalue)
    except ValueError:
        w_p = np.nan
    per_run.append({
        "run": r["run"], "arm": r["arm"], "train_seed": r.get("seed", "?"),
        "mean_rl": d["yield_kg"].mean(), "mean_delta": dl.mean(),
        "ci95_lo": dl.mean() - 1.96 * dl.std(ddof=1) / np.sqrt(len(dl)),
        "ci95_hi": dl.mean() + 1.96 * dl.std(ddof=1) / np.sqrt(len(dl)),
        "wilcoxon_p": w_p,
        "ttest_p": float(stats.ttest_rel(d["yield_kg"], d["yield_recipe"]).pvalue),
        "winrate": float((dl > 0).mean()),
        "collapse_rate": float((d["yield_kg"] < FAILED_KG).mean()),
    })
per_run = pd.DataFrame(per_run)
print("=== per run (unit = evaluation batch; describes THIS policy) ===")
display(per_run.round(3))

print("\n=== across runs (unit = training run; describes THE METHOD) ===")
for arm, g in per_run.groupby("arm"):
    md_ = g["mean_delta"].values
    if len(md_) > 1:
        t = stats.ttest_1samp(md_, 0.0)
        print(f"{arm}: n_runs={len(md_)}  mean delta {md_.mean():+8.1f} kg  "
              f"sd {md_.std(ddof=1):7.1f}  p={t.pvalue:.3f}")
    else:
        print(f"{arm}: n_runs=1 -> NO across-run inference possible. "
              f"mean delta {md_[0]:+.1f} kg is a single draw; run more training seeds.")

## E.3 — Policy progression, per training seed

Each trial's saved policy evaluated on `PROG_SEEDS`, paired against the recipe on the same seeds.

`log.pkl` records the yield of each trial's own *training* episode, but that number is unusable as a
learning curve: every episode runs on a different realisation, so it confounds policy quality with
seed luck. Re-rolling every checkpoint on a *fixed* seed set is what isolates the policy.

Kept to three seeds because this is `n_runs x n_trials x n_seeds` batches.

In [ ]:
prog_recipe_rows = [{"run": "_recipe", "trial": 0, "seed": s, "kind": "recipe_prog"}
                    for s in PROG_SEEDS]
prog_recipe = _cached(prog_recipe_rows, lambda r: _batch(r["seed"], policy=None, pid_baseline=True))
prog_recipe_y = prog_recipe.set_index("seed")["yield_kg"]

prog_rows = [{"run": r["run"], "trial": k, "seed": s, "kind": "rl_prog"}
             for r in RUNS for k in range(1, r["n_trials"] + 1) for s in PROG_SEEDS]
prog = _cached(prog_rows, lambda r: _batch(r["seed"], policy=_load_policy(_paths[r["run"]], r["trial"])))
prog["yield_recipe"] = prog["seed"].map(prog_recipe_y)
prog["delta"] = prog["yield_kg"] - prog["yield_recipe"]
prog["arm"] = prog["run"].map({r["run"]: r["arm"] for r in RUNS})

n = len(RUNS)
ncol = min(4, n); nrow = int(np.ceil(n / ncol))
fig, ax = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.0 * nrow), squeeze=False, sharey=True)
for i, r in enumerate(RUNS):
    a = ax[i // ncol, i % ncol]
    g = prog[prog.run == r["run"]].groupby("trial")["yield_kg"]
    mu, sd, tr = g.mean(), g.std(ddof=1), sorted(g.groups)
    a.plot(tr, mu, "o-", color="C0", lw=1.8, ms=5, label="RL policy")
    a.fill_between(tr, mu - sd, mu + sd, color="C0", alpha=.18, lw=0)
    a.axhline(prog_recipe_y.mean(), color="C3", ls="--", lw=1.6, label="recipe (same seeds)")
    a.axhline(FAILED_KG, color="0.6", ls=":", lw=1.2, label=f"collapse ({FAILED_KG:g} kg)")
    a.set_title(f"{r['run']}\n{r['arm']}", fontsize=8)
    a.set_xlabel("trial", fontsize=8); a.set_xticks(tr)
    a.grid(alpha=.25, lw=.5); a.tick_params(labelsize=7)
    if i % ncol == 0:
        a.set_ylabel("held-out yield (kg)", fontsize=8)
for j in range(n, nrow * ncol):
    ax[j // ncol, j % ncol].axis("off")
h, l = ax[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(.5, -.02))
fig.suptitle(f"E.3 - policy progression on fixed held-out seeds {PROG_SEEDS} "
             "(band = +/-1 sd across those seeds)", fontsize=10)
fig.tight_layout(); plt.show()

print("A flat or falling curve means the extra trials bought nothing on unseen batches, regardless")
print("of what the imagined cost did during optimisation.")

## E.4 — Mean held-out comparison, and the distribution behind it

The bar chart is the headline number. The two panels beside it exist because the mean alone is
misleading when outcomes are bimodal: the per-seed paired deltas show *how often* RL wins rather
than by how much on average, and a single collapsed batch is visible rather than averaged away.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))

# --- means, with the recipe as the paired reference -----------------------------------
a = ax[0]
labels = ["recipe"] + [r["run"] for r in RUNS]
means = [recipe_y.mean()] + [rl_df[rl_df.run == r["run"]]["yield_kg"].mean() for r in RUNS]
sems = [recipe_y.std(ddof=1) / np.sqrt(len(recipe_y))] + \
       [rl_df[rl_df.run == r["run"]]["yield_kg"].std(ddof=1) / np.sqrt(len(EVAL_SEEDS)) for r in RUNS]
cols = ["C3"] + ["C0"] * len(RUNS)
a.bar(range(len(labels)), means, yerr=sems, capsize=5, color=cols, width=.65)
for i, v in enumerate(means):
    a.annotate(f"{v:.0f}", (i, v), ha="center", fontsize=8, xytext=(0, 4),
               textcoords="offset points", color="0.3")
a.set_xticks(range(len(labels)))
a.set_xticklabels(labels, fontsize=7, rotation=30, ha="right")
a.set_ylabel("mean held-out yield (kg)", fontsize=9)
a.set_title(f"mean yield, n={len(EVAL_SEEDS)} paired seeds (error bar = SEM)", fontsize=9)
a.grid(alpha=.25, lw=.5, axis="y")

# --- per-seed paired delta: one dot per held-out batch ---------------------------------
a = ax[1]
for i, r in enumerate(RUNS):
    d = rl_df[rl_df.run == r["run"]]["delta"].values
    a.plot(np.full_like(d, i, dtype=float) + np.linspace(-.15, .15, len(d)), d,
           ls="none", marker="o", ms=4.5, color="C0", alpha=.75)
    a.plot([i - .25, i + .25], [d.mean()] * 2, color="C0", lw=2.4)
a.axhline(0, color="C3", ls="--", lw=1.6, label="recipe parity")
a.set_xticks(range(len(RUNS)))
a.set_xticklabels([r["run"] for r in RUNS], fontsize=7, rotation=30, ha="right")
a.set_ylabel("yield - recipe, same seed (kg)", fontsize=9)
a.set_title("paired delta per held-out seed (bar = mean)", fontsize=9)
a.grid(alpha=.25, lw=.5, axis="y"); a.legend(fontsize=7.5)

# --- pooled delta distribution ---------------------------------------------------------
a = ax[2]
a.hist(rl_df["delta"].values, bins=max(6, len(rl_df) // 4), color="C0", alpha=.85, edgecolor="k")
a.axvline(0, color="C3", ls="--", lw=1.6)
a.axvline(rl_df["delta"].mean(), color="k", ls="-", lw=1.6,
          label=f"mean {rl_df['delta'].mean():+.0f} kg")
a.set_xlabel("paired delta (kg)", fontsize=9); a.set_ylabel("count", fontsize=9)
a.set_title("pooled delta distribution", fontsize=9)
a.grid(alpha=.25, lw=.5, axis="y"); a.legend(fontsize=7.5)

fig.suptitle("E.4 - held-out yield: RL vs recipe (paired on identical simulator seeds)", fontsize=10)
fig.tight_layout(); plt.show()

## E.5 — Reliability: collapse rate, tail risk, and the viscosity mechanism

Mean yield is the wrong summary for a bimodal outcome. A policy averaging 3500 kg by producing
mostly 3900 kg with occasional 1000 kg collapses is a different proposition from one that reliably
produces 3500 kg, and only the second is deployable.

`CVaR@10%` is the mean of the worst decile — what you get on a bad day.

The scatter tests the known failure mechanism: sustained overfeeding thickens the broth, oxygen
transfer fails, and penicillin degrades instead of accumulating. Every collapse observed so far sat
above ~150 cP while every healthy batch stayed below ~111 cP.

In [ ]:
def _cvar(x, q=0.10):
    x = np.sort(np.asarray(x)); k = max(1, int(np.ceil(q * len(x))))
    return float(x[:k].mean())

rel = []
for label, y, v in ([("recipe", recipe_df["yield_kg"].values, recipe_df["max_visc"].values)] +
                    [(r["run"], rl_df[rl_df.run == r["run"]]["yield_kg"].values,
                      rl_df[rl_df.run == r["run"]]["max_visc"].values) for r in RUNS]):
    rel.append({"arm": label, "mean_kg": y.mean(), "sd_kg": y.std(ddof=1), "worst_kg": y.min(),
                "cvar10_kg": _cvar(y), "collapse_rate": float((y < FAILED_KG).mean()),
                "max_visc_p95": float(np.percentile(v, 95)),
                "visc_over_max": float((v > VISC_MAX).mean())})
rel = pd.DataFrame(rel)
display(rel.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
a = ax[0]
a.scatter(recipe_df["max_visc"], recipe_df["yield_kg"], s=52, color="C3", marker="s",
          label="recipe", zorder=3, edgecolor="white", lw=1)
a.scatter(rl_df["max_visc"], rl_df["yield_kg"], s=52, color="C0",
          label="RL", zorder=3, edgecolor="white", lw=1)
a.axvline(VISC_MAX, color="0.5", ls=":", lw=1.4)
a.annotate("VISC_MAX", (VISC_MAX, a.get_ylim()[1]), fontsize=7, rotation=90,
           xytext=(3, -46), textcoords="offset points", color="0.4")
a.axhline(FAILED_KG, color="0.6", ls="--", lw=1.2)
a.annotate(f"collapse ({FAILED_KG:g} kg)", (a.get_xlim()[0], FAILED_KG), fontsize=7,
           xytext=(4, 4), textcoords="offset points", color="0.4")
a.set_xlabel("peak broth viscosity (cP)", fontsize=9)
a.set_ylabel("batch yield (kg)", fontsize=9)
a.set_title("the failure mechanism: yield vs peak viscosity", fontsize=9)
a.grid(alpha=.25, lw=.5); a.legend(fontsize=8)

a = ax[1]
_lbl = list(rel["arm"])
a.bar(range(len(rel)), rel["cvar10_kg"], color=["C3"] + ["C0"] * len(RUNS), width=.6,
      label="CVaR@10% (worst decile)")
a.plot(range(len(rel)), rel["mean_kg"], "o", color="k", ms=6, label="mean")
a.set_xticks(range(len(rel))); a.set_xticklabels(_lbl, fontsize=7, rotation=30, ha="right")
a.set_ylabel("yield (kg)", fontsize=9)
a.set_title("mean vs bad-day outcome — a wide gap means unreliable", fontsize=9)
a.grid(alpha=.25, lw=.5, axis="y"); a.legend(fontsize=8)

fig.suptitle("E.5 - reliability and the viscosity failure mode", fontsize=10)
fig.tight_layout(); plt.show()

if len(rl_df):
    hi = rl_df[rl_df.max_visc > 150]
    print(f"RL batches above 150 cP: {len(hi)}/{len(rl_df)}"
          + (f"  -> mean yield {hi['yield_kg'].mean():.0f} kg vs "
             f"{rl_df[rl_df.max_visc <= 150]['yield_kg'].mean():.0f} kg below" if len(hi) else ""))

## E.6 — Substrate efficiency

Yield alone can be bought by feeding more sugar. `Fs` is the only actuator the agent controls, so a
gain that costs proportionally more substrate is not obviously a gain at all — this panel reports
kg penicillin per kg sugar fed, paired on the same seeds as everything above.

In [ ]:
rl_df["eff"] = rl_df["yield_kg"] / rl_df["sugar_kg"]
recipe_df["eff"] = recipe_df["yield_kg"] / recipe_df["sugar_kg"]
rec_eff = recipe_df.set_index("seed")["eff"]
rl_df["eff_recipe"] = rl_df["seed"].map(rec_eff)
rl_df["sugar_recipe"] = rl_df["seed"].map(recipe_df.set_index("seed")["sugar_kg"])

eff = (rl_df.groupby("run")
       .agg(sugar_kg=("sugar_kg", "mean"), sugar_recipe=("sugar_recipe", "mean"),
            eff=("eff", "mean"), eff_recipe=("eff_recipe", "mean"),
            yield_kg=("yield_kg", "mean"))
       .reset_index())
eff["sugar_vs_recipe_%"] = 100 * (eff.sugar_kg / eff.sugar_recipe - 1)
eff["eff_vs_recipe_%"] = 100 * (eff.eff / eff.eff_recipe - 1)
display(eff.round(3))

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.axhline(0, color="0.6", lw=1); ax.axvline(0, color="0.6", lw=1)
ax.scatter(eff["sugar_vs_recipe_%"],
           100 * (eff.yield_kg / rl_df.groupby("run")["yield_recipe"].mean().values - 1),
           s=70, color="C0", zorder=3, edgecolor="white", lw=1.2)
for _, r in eff.iterrows():
    ax.annotate(r["run"], (r["sugar_vs_recipe_%"], 0), fontsize=7, color="0.35",
                xytext=(4, 4), textcoords="offset points")
lim = max(2.0, np.abs(eff["sugar_vs_recipe_%"]).max() * 1.2)
xs = np.linspace(-lim, lim, 50)
ax.plot(xs, xs, color="0.55", ls="--", lw=1.2, label="break-even (yield tracks sugar)")
ax.set_xlabel("extra sugar fed vs recipe (%)", fontsize=9)
ax.set_ylabel("extra yield vs recipe (%)", fontsize=9)
ax.set_title("E.6 - is the gain bought with sugar?\nabove the dashed line = genuine efficiency gain",
             fontsize=9)
ax.grid(alpha=.25, lw=.5); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Reading this notebook

1. **E.2 across-runs** is the claim that generalises. A single training run cannot support a
   statement about the method, no matter how tight its per-run CI looks.
2. **E.4** is the headline number, but read it next to **E.5**: a mean that hides a 20% collapse
   rate is not a result you can deploy.
3. **E.3** tells you whether training helped at all on unseen batches — independently of what the
   imagined cost did during optimisation.
4. **E.6** guards against the trivial win of simply feeding more.